In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os

In [ ]:
import os
import pandas as pd

# Função para classificar valores nas faixas
def substituir_valor(valor):
    if valor < 200000000:
        return "r"
    elif 200000000 <= valor < 500000000:
        return "o"
    elif 500000000 <= valor < 800000000:
        return "y"
    elif 800000000 <= valor < 1000000000:
        return "b"
    else:
        return "g"

# Caminho da pasta onde estão os arquivos CSV
caminho_pasta = '../../results/regression/predictions-regression-bysource'

# Listar os arquivos da pasta
arquivos = os.listdir(caminho_pasta)

# Inicializar lista para armazenar os resultados
resultados = []

# Iterar sobre os arquivos
for arquivo in arquivos:
    if arquivo.endswith(".csv") and arquivo.startswith("Vazao_bbr"):
        # Caminho completo do arquivo
        caminho_arquivo = os.path.join(caminho_pasta, arquivo)
        
        # Carregar o arquivo CSV
        df = pd.read_csv(caminho_arquivo)
        
        # Verificar se a coluna 'y_test' está no DataFrame
        if 'y_test' in df.columns:
            # Classificar os valores de y_test nas faixas
            df['faixa_y_test'] = df['y_test'].apply(substituir_valor)
            
            # Iterar sobre as previsões de cada modelo
            for modelo in df.columns:
                if modelo.startswith('y_predict'):  # Ignorar a coluna 'y_test'
                    # Classificar os valores das previsões nas faixas
                    coluna_faixa = f'faixa_{modelo}'
                    df[coluna_faixa] = df[modelo].apply(substituir_valor)
                    
                    # Verificar se y_test e y_pred estão na mesma faixa
                    coluna_comparacao = f'mesma_faixa_{modelo}'
                    df[coluna_comparacao] = df['faixa_y_test'] == df[coluna_faixa]
                    
                    # Contabilizar quantos estão na mesma faixa
                    mesma_faixa_count = df[coluna_comparacao].sum()
                    total_count = df.shape[0]
                    
                    # Armazenar os resultados para cada modelo
                    resultados.append({
                        "arquivo": arquivo.split("_")[2].split(".")[0],# + " "+arquivo.split("_")[1],
                        "modelo": modelo,
                        "mesma_faixa_count": mesma_faixa_count,
                        "total_count": total_count,
                        "percentual_misma_faixa": (mesma_faixa_count / total_count) * 100
                    })

# Converter resultados para um DataFrame
df_resultados = pd.DataFrame(resultados)

In [ ]:
df_resultados

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def substituir_valor(valor):
    if valor < 200000000:
        return "r"
    elif 200000000 <= valor < 500000000:
        return "o"
    elif 500000000 <= valor < 800000000:
        return "y"
    elif 800000000 <= valor < 1000000000:
        return "b"
    else:
        return "g"

#5.000.000 ou 10.000.000
# def substituir_valor_fuzzy(valor, janela=10000000):
#     if valor < 200000000 + janela:
#         return "r"
#     elif 200000000 <= valor < 500000000 - janela:
#         return "o"
#     elif 500000000 - janela <= valor < 800000000 + janela:
#         return "y"
#     elif 800000000 <= valor < 1000000000 - janela:
#         return "b"
#     else:
#         return "g"

def process_files(folder_path, excluded_sources=None, excluded_models=None):
    if excluded_sources is None:
        excluded_sources = []
    if excluded_models is None:
        excluded_models = []

    results = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.csv') and file_name.startswith('Vazao_bbr_'):
            source = file_name.replace('.csv', '').split('_')[2].upper()  
            if source in excluded_sources:
                continue
            file_path = os.path.join(folder_path, file_name)

            df = pd.read_csv(file_path)

            if 'y_test' not in df.columns:
                print(f"`y_test` não encontrado em {file_name}, ignorando.")
                continue

            df['y_test_cat'] = df['y_test'].apply(substituir_valor)
            for col in df.columns:
                if col.startswith('y_pred'):
                    if col.split('_')[2] in excluded_models:
                        continue
                    df[f'{col}_cat'] = df[col].apply(substituir_valor)
                    matches = (df['y_test_cat'] == df[f'{col}_cat']).sum()
                    total = len(df)
                    accuracy = (matches / total) * 100
                    results.append({
                        'Source': source,
                        'Model': col.split('_')[2],
                        'Accuracy (%)': accuracy
                    })
    return pd.DataFrame(results)

def plot_results(results_df):

    plt.figure(figsize=(10, 15))  

    ax = sns.barplot(
        width=0.8,  
        data=results_df,
        y='Source',  
        x='Accuracy (%)', 
        hue='Model', 
        palette='viridis',
        dodge=True, 
        edgecolor='black' 
    )

    for bar in ax.patches:
        bar_width = bar.get_width()
        bar_height = bar.get_height()
        bar_x = bar.get_x()
        bar_y = bar.get_y()
        ax.plot(
            [bar_x + bar_width] * 2,  
            [bar_y, bar_y + bar_height],
            color='black',
            lw=1.5
        )
    for x in ax.get_xticks():
        ax.axvline(x=x, color='gray', linestyle='--', linewidth=1)  

    plt.title('Comparação de Acertos por Modelo e Fonte', fontsize=16)
    plt.ylabel('Fonte', fontsize=14)
    plt.xlabel('Porcentagem de Acertos (%)', fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    ax.set_xlim(0, 100)

    ax.set_yticks(ax.get_yticks()) 

    plt.legend(
        title='Modelos de Regressao',
        fontsize=12,
        title_fontsize=14,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.05),  
        ncol=3  
    )

    plt.tight_layout()
    plt.show()


folder_path = '../../results/regression/predictions-regression-bysource'

excluded_sources = ['DF']  
excluded_models = ['PolynomialRegression', 'AdaBoostRegressor', 'ElasticNet', 
                   'LinearRegression', 'MLPRegressor', 'SVR', 'KNeighborsRegressor'] 

results_df = process_files(folder_path, excluded_sources=excluded_sources, excluded_models=excluded_models)

plot_results(results_df)


In [ ]:
dropado = df_resultados.drop(columns = ['arquivo', 'modelo'])

In [ ]:
mean = dropado.mean()

In [ ]:
mean

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Carregar o arquivo CSV
df = df_resultados
# Configurar o estilo do gráfico
sns.set(style="whitegrid")

# Plotar o gráfico de barras
plt.figure(figsize=(12, 6))

# Criar o gráfico de barras agrupado por 'arquivo' (source) e os modelos
sns.barplot(data=df, x='arquivo', y='percentual_misma_faixa', hue='modelo', ci=None)

# Título e rótulos do gráfico
plt.title('Percentual de Previsões na Mesma Faixa por Fonte (Source) e Modelo', fontsize=14)
plt.xlabel('Fonte (Source)', fontsize=12)
plt.ylabel('Percentual na Mesma Faixa (%)', fontsize=12)

# Rotacionar os rótulos do eixo X para melhor visualização
plt.xticks(rotation=0)

# Exibir a legenda
plt.legend(title='Modelo', bbox_to_anchor=(1.05, 1), loc='upper left')

# Exibir o gráfico
plt.tight_layout()
plt.show()
